In [3]:
import pandas as pd
import geopandas as gpd
from pandas.api.types import CategoricalDtype
from shapely.geometry import Point

# --- PATHS ---
COMPANY_CSV = r"C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\organized\4_2023_optionsMapped.csv"
DEGURBA_SHP = r"C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\2024\Data\AuxData\DGURBA_RG_01M_2021_4258\DGURBA_RG_01M_2021_4258.shp"
OUTPUT_CSV  = r"C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\organized\5_2023_urbanCategoryFound.csv"

# coordinate columns in your CSV
LON_OFFICE = "lon_office"
LAT_OFFICE = "lat_office"
LON_HOME   = "lon_home"
LAT_HOME   = "lat_home"

# output column names requested
OUT_OFFICE = "office_location_type"
OUT_HOME   = "home_location_type"

# ---------- helpers ----------
def normalize_deg_label_from_dgurba(v):
    """
    Map DGURBA values to textual class:
    Cities (Urban) / Towns and suburbs (Peri-urban) / Rural areas (Non-urban)
    Handles numeric and alphabetic encodings commonly found in DGURBA/DEGURBA layers.
    """
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip().lower()

    # numeric encodings
    if s in {"1","1.0"}: return "Cities (Urban)"
    if s in {"2","2.0"}: return "Towns and suburbs (Peri-urban)"
    if s in {"3","3.0"}: return "Rural areas (Non-urban)"

    # letter encodings
    if s in {"d","c"}: return "Cities (Urban)"
    if s in {"i","t"}: return "Towns and suburbs (Peri-urban)"
    if s in {"r"}:     return "Rural areas (Non-urban)"

    # word-ish encodings
    if "city" in s or "cities" in s or "densely" in s or "urban centre" in s:
        return "Cities (Urban)"
    if "town" in s or "suburb" in s or "intermediate" in s:
        return "Towns and suburbs (Peri-urban)"
    if "rural" in s or "thinly" in s:
        return "Rural areas (Non-urban)"

    return None

bucket_map = {
    "Cities (Urban)": "urban",
    "Towns and suburbs (Peri-urban)": "peri_urban",
    "Rural areas (Non-urban)": "non_urban",
}
cat = CategoricalDtype(["urban", "peri_urban", "non_urban"], ordered=True)

def build_points_gdf(df: pd.DataFrame, lon_col: str, lat_col: str, crs="EPSG:4326") -> gpd.GeoDataFrame:
    """
    Create a GeoDataFrame of points from lon/lat columns.
    Drops rows with missing/invalid coords but preserves original index for merging back.
    """
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    lat = pd.to_numeric(df[lat_col], errors="coerce")

    valid = lon.notna() & lat.notna() & lon.between(-180, 180) & lat.between(-90, 90)

    pts = df.loc[valid].copy()
    pts["geometry"] = [Point(xy) for xy in zip(lon.loc[valid], lat.loc[valid])]
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=crs)

def classify_points(points_gdf: gpd.GeoDataFrame, deg_polys: gpd.GeoDataFrame) -> pd.Series:
    """
    Spatial join points -> polygons, then map DGURBA to your bucket labels.
    Returns a Series indexed like points_gdf (original df index).
    """
    if points_gdf.empty:
        return pd.Series(dtype="object")

    if points_gdf.crs != deg_polys.crs:
        points_gdf = points_gdf.to_crs(deg_polys.crs)

    subset = deg_polys[["DGURBA", "geometry"]].copy()
    joined = gpd.sjoin(points_gdf, subset, how="left", predicate="within")

    joined["DEGURBA_L1_TEXT"] = joined["DGURBA"].apply(normalize_deg_label_from_dgurba)
    loc_class = joined["DEGURBA_L1_TEXT"].map(bucket_map).astype(cat)

    loc_class.index = joined.index
    return loc_class

# ---------- 1) Read input ----------
df = pd.read_csv(COMPANY_CSV, encoding="ISO-8859-1")

# ---------- 2) Read DEGURBA polygons ----------
gdf_deg = gpd.read_file(DEGURBA_SHP)
assert "DGURBA" in gdf_deg.columns, "DGURBA column not found in polygon layer."

# ---------- 3) Build office + home point layers ----------
gdf_office = build_points_gdf(df, LON_OFFICE, LAT_OFFICE)
gdf_home   = build_points_gdf(df, LON_HOME,   LAT_HOME)

print(f"Office points with valid coords: {len(gdf_office)}/{len(df)}")
print(f"Home points with valid coords:   {len(gdf_home)}/{len(df)}")

# ---------- 4) Classify office + home ----------
office_class = classify_points(gdf_office, gdf_deg)
home_class   = classify_points(gdf_home,   gdf_deg)

# ---------- 5) Merge back, REMOVE missing home_location_type, & save ----------
df_out = df.copy()
df_out[OUT_OFFICE] = pd.NA
df_out[OUT_HOME]   = pd.NA

df_out.loc[office_class.index, OUT_OFFICE] = office_class.astype("object").values
df_out.loc[home_class.index,   OUT_HOME]   = home_class.astype("object").values

# remove rows where home_location_type is missing
before = len(df_out)
df_out = df_out[df_out[OUT_HOME].notna()].copy()
after = len(df_out)

df_out.to_csv(OUTPUT_CSV, index=False, encoding="ISO-8859-1")
print(f"✅ Saved -> {OUTPUT_CSV}")
print(f"Removed rows with missing {OUT_HOME}: {before - after} (kept {after})")

print("\nOffice location type counts:")
print(df_out[OUT_OFFICE].value_counts(dropna=False))

print("\nHome location type counts:")
print(df_out[OUT_HOME].value_counts(dropna=False))


Office points with valid coords: 1527/1527
Home points with valid coords:   1527/1527
✅ Saved -> C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\organized\5_2023_urbanCategoryFound.csv
Removed rows with missing home_location_type: 1 (kept 1526)

Office location type counts:
office_location_type
urban         1453
peri_urban      73
Name: count, dtype: int64

Home location type counts:
home_location_type
urban         1096
peri_urban     387
non_urban       43
Name: count, dtype: int64
